# Aero3D · Stage 04 — Camera pose (COLMAP) + depth (Depth Anything V2)

Runs on a free Colab GPU. **Before you start:** *Runtime → Change runtime type → T4 GPU*.

**Inputs:** the package `aero3d_stage04_<project>.zip` downloaded from the Aero3D app (Pose & Depth page).
**Output:** `stage04_results.zip`, which you import back into the app.

| Step | What happens | Typical time (≈150 keyframes) |
|---|---|---|
| 3 | COLMAP: features → matching → incremental SfM + bundle adjustment → poses, intrinsics, sparse points | 3–20 min |
| 4 | Depth Anything V2 depth map per registered image, aligned to the COLMAP points | 2–6 min |

Notes
- Poses and depth are in **COLMAP's own arbitrary scale and orientation, not metres**. Metric scale comes from georeferencing later.
- `Depth-Anything-V2-Large` is licensed **CC-BY-NC-4.0** (non-commercial). Use `...-Base-hf` or `...-Small-hf` (Apache-2.0) if that matters.
- COLMAP runs in its own virtual environment on purpose: the CUDA build of `pycolmap` pins CUDA libraries that conflict with PyTorch.

## 1 · Check the GPU

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print(out or "NO GPU DETECTED -> Runtime > Change runtime type > T4 GPU (CPU works too, but is much slower)")

## 2 · Upload the package

In [ ]:
import json, os, shutil, zipfile
from google.colab import files

PKG, WORK = "/content/pkg", "/content/work"
shutil.rmtree(PKG, ignore_errors=True)

# Faster for big packages: mount Drive, put the zip there and set zip_path to it instead of uploading.
uploaded = files.upload()  # choose aero3d_stage04_<project>.zip
zip_path = next(iter(uploaded))
with zipfile.ZipFile(zip_path) as z:
    z.extractall(PKG)

manifest = json.load(open(f"{PKG}/manifest.json"))
assert manifest.get("format") == "aero3d-stage04", "This is not an Aero3D Stage 04 package"
print("Project  :", manifest["project"])
print("Keyframes:", len(manifest["keyframes"]), "| video:", manifest["video"].get("resolution"), "@", manifest["video"].get("fps"), "fps")
for w in manifest.get("warnings", []):
    print("WARNING  :", w)

## 3 · COLMAP (pose + intrinsics + sparse points)

In [ ]:
%%bash
set -e
apt-get -qq install -y python3-venv > /dev/null 2>&1 || true
rm -rf /content/colmap_env
python3 -m venv /content/colmap_env
/content/colmap_env/bin/pip -q install --upgrade pip
# GPU build if it installs, otherwise the CPU build
/content/colmap_env/bin/pip -q install pycolmap-cuda12 || { echo "CUDA build unavailable -> installing CPU pycolmap"; /content/colmap_env/bin/pip -q install pycolmap; }
/content/colmap_env/bin/python -c "import pycolmap; print('pycolmap', pycolmap.__version__, '| CUDA build:', getattr(pycolmap, 'has_cuda', False))"

In [ ]:
import subprocess

PY = "/content/colmap_env/bin/python"

def run(cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    return p.wait(), "".join(lines)

def colmap(device):
    return run([PY, f"{PKG}/colmap_pipeline.py", "--images", f"{PKG}/images", "--out", WORK, "--device", device])

rc, log = colmap("auto")
if rc != 0 and "could not build any reconstruction" not in log:
    print("\n>>> GPU run failed; retrying on CPU ...\n")
    rc, log = colmap("cpu")
assert rc == 0, "COLMAP failed - read the messages above (usually: too little overlap between keyframes)"

## 4 · Depth (optional — skip this step if you only need camera poses)
Pick a model: `Large` is the most accurate. `Base` / `Small` are faster and Apache-2.0.

In [ ]:
DEPTH_MODEL = "depth-anything/Depth-Anything-V2-Large-hf"   # or ...-Base-hf / ...-Small-hf

In [ ]:
%%bash -s "$DEPTH_MODEL"
pip -q install "transformers>=4.45" accelerate
python /content/pkg/depth_pipeline.py --images /content/pkg/images --work /content/work --model "$1"

## 5 · Download the results

In [ ]:
import json, os, shutil

RES = "/content/stage04_results"
shutil.rmtree(RES, ignore_errors=True)
os.makedirs(RES)
shutil.copytree(f"{WORK}/model", f"{RES}/model")
for d in ("depth", "depth_preview"):
    if os.path.isdir(f"{WORK}/{d}"):
        shutil.copytree(f"{WORK}/{d}", f"{RES}/{d}")
for f in ("colmap_report.json", "depth_report.json"):
    if os.path.isfile(f"{WORK}/{f}"):
        shutil.copy(f"{WORK}/{f}", f"{RES}/{f}")
shutil.copy(f"{PKG}/manifest.json", f"{RES}/manifest.json")   # lets the app verify the results match the project
shutil.make_archive("/content/stage04_results", "zip", RES)

rep = json.load(open(f"{WORK}/colmap_report.json"))
print(f"Registered {rep['num_images_registered']}/{rep['num_images_input']} images | "
      f"{rep['num_points3D']} points | reprojection error {rep['mean_reprojection_error_px']} px")
for w in rep["warnings"]:
    print("WARNING:", w)

from google.colab import files
files.download("/content/stage04_results.zip")